# Homework 5

Suppose $S_1(t)$ and $S_2(t)$ are independent stock processes with equal volatility. $S_1(0)=S_2(0)=100.$
The contract at maturity pays the value of the most expensive stock: $V(T)=\max_i\{S_1(T),S_2(T)\}.$
* What is the formula for the fair value of the contract, $V(0)$?         
* Derive the partial differential equation for this contract              
* Solve the resultant PDE using Finite Difference Method                   
* Compare the results obtained through solving the PDE with Monte Carlo Method

I also attach a pdf written by me in LaTeX, where I wrote down the detailed calculations and how I got the formulas which I use here in the code.

## Part a)

The formula for $V(0)$ is the following:
$$
V(0)=200\cdot\Phi\left(\sigma\cdot \sqrt{\frac{T}{2}}\right)
$$
I do a calculation, so that after solving the PDE we can compare the results.

In [1]:
# Import everything from the 07_part1_Finite_... notebook
from typing import Callable
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.stats import norm
from scipy import linalg
import numpy as np

In [2]:
T = 2.0
SIGMA = 0.12
V0_exact = 200*norm.cdf(SIGMA*np.sqrt(T/2))
print(f"V(0) = {V0_exact}")
# This will be our "reference" value

V(0) = 109.55168520411678


## Part b)

In the LaTeX file.

## Part c)

I made some changes in the PDE from Part b), so that the 1D grid will be sufficient, and only small changes are necessary in the codes presented in the class' notebook.<br>
The goal was to transform the PDE into the following form:
$$
-\frac{\partial V(x,t)}{\partial t} = \left[-r + \left(r-\frac{\sigma_c^2}{2}\right)\frac{\partial}{\partial x} + \frac{\sigma_c^2}{2}\frac{\partial ^2}{\partial x^2}\right]V(x,t)
$$
In the pdf I also wrote down the steps, and the new names of the variables.<br>
The key differences are that we choose $r=0.0$, and substitute in the place of $\sigma_c$ our new parameter, $\sqrt{2}\sigma$.


In [3]:
def get_d1(n):
    #First and last rows are incorrect, but they are given by boundary condition
    return 0.5*(np.diag(np.ones(n-1), 1)-np.diag(np.ones(n-1), -1))

def get_d2(n):
    #First and last rows are incorrect, but they are given by boundary condition
    return np.diag(np.ones(n-1), 1)+np.diag(-2.*np.ones(n))+np.diag(np.ones(n-1), -1)

def get_a_max_option(n, dx, sig):
  # Our PDE operator: A = - sigma^2 * D1 + sigma^2 * D2
  d1 = (sig**2) / dx * get_d1(n)
  d2 = (sig**2) / (dx**2) * get_d2(n)
  return -d1 + d2

In [4]:
# The new boundary function
def max_option_boundary(grid: np.ndarray, dx: float, dt: float):
    """
    Boundary and terminal conditions for w(y, t) where y = log(S1/S2)
    and V(S1, S2, t) = S2 * w(y, t).
    Assume grid is linearly spaced and grid[:,-1] is t=T.
    """
    nx, nt = grid.shape
    # Bottom boundary: S1/S2 -> 1.0
    grid[0, :]  = 1.0
    
    # Top boundary: S1 >> S2, S1/S2 -> infty
    grid[-1, :] = np.exp(y_max)
    
    # Terminal condition at t = T: w(y, T) = max(exp(y), 1)
    y_vec = np.linspace(y_min, y_max, n_y)
    grid[:, -1] = np.maximum(np.exp(y_vec), 1.0)
    
    return grid

In [5]:
def create_and_run_CN_max_option(n_y, n_t):
    theta=0.5
    print(f"theta={theta}")
    dy = (y_max-y_min)/(n_y-1)
    dt = T/(n_t-1)
    
    cn_grid = np.zeros((n_y, n_t))
    cn_grid = max_option_boundary(cn_grid, dy, dt)

    A_max = get_a_max_option(n_y, dy, SIGMA)
    
    matrix1 = (1.-theta)*dt*A_max-np.diag(np.ones(n_y))
    matrix2 = np.diag(np.ones(n_y)) + theta*dt*A_max
    
    for n in range(1, n_t)[::-1]:
        boundary = np.dot(matrix1, cn_grid[:,n-1])
        cn_grid[1:-1,n-1] = linalg.solve(matrix1[1:-1,1:-1], -np.dot(matrix2, cn_grid)[1:-1,n]-boundary[1:-1])
        
    return cn_grid

At $t=0$, we have $S_1(0)=S_2(0)=100$, so 
$$
y_0=\ln\left(\frac{100}{100}\right)=\ln(1)=0 
$$

In [6]:
# We run the solver with odd number of spatial steps, so y_0 is in the middle
n_y, n_t = 501, 501     # as we increase these numbers, the result gets closer to the analytical solution, 
                        # but also it takes more time
S1_0, S2_0 = 100, 100
y_min = -5
y_max = 5

GRID = create_and_run_CN_max_option(n_y, n_t)
# Middle row index corresponds to y = log(100/100) = 0
mid_idx = n_y // 2

# Multiply w(0, 0) by S2(0) = 100 to get V(0)
V0_pde = S2_0 * GRID[mid_idx, 0]

print(f"PDE Price V(0):      {V0_pde:.4f}")
print(f"Analytical V(0):     {V0_exact:.4f}")


theta=0.5
PDE Price V(0):      109.5432
Analytical V(0):     109.5517


## Part d)

In the pdf I also wrote about how I got the formulas for the code, and what theorem I found on Wikipedia which produces these thoughts.

In [7]:
num_of_paths = 100000

# Here we used the 1D PDE again
# Based on numerical_method.py generate_std_norm:
np.random.seed(1)
rnd1 = np.random.standard_normal(size=(int(num_of_paths / 2), 1)) # 50000 row and one column
rnd = np.concatenate((rnd1, -rnd1), axis=0) # the -1 times value of each (upward shock, downward shock)
rnd = (rnd - np.mean(rnd)) / np.std(rnd, ddof=1) # the sample of random numbers should have 0 mean and 1 variance by the
                                                 # theoretical solution

# Parameters according to the 1D PDE: x_0 = 1, vol_eff = sqrt(2)*SIGMA
x_0 = S1_0 / S2_0
vol_eff = np.sqrt(2.0) * SIGMA

# The solution which we have to simulate
x_T = x_0 * np.exp(-(SIGMA**2) * T + vol_eff * rnd[:, 0] * np.sqrt(T))

# Payment u(x, T) = max(x_T, 1.0), then multiplying with S2_0: V(0) = S2_0 * E[max(x_T, 1)]
u_0_mc = np.mean(np.maximum(x_T, 1.0))
V0_mc_1d = S2_0 * u_0_mc

print(f"Monte Carlo 1D PDE Price V(0): {V0_mc_1d:.4f}")
print(f"PDE Price V(0):                {V0_pde:.4f}")
print(f"Analytical V(0):               {V0_exact:.4f}")


Monte Carlo 1D PDE Price V(0): 109.5575
PDE Price V(0):                109.5432
Analytical V(0):               109.5517
